In [38]:
from atomica.models import MultiClassClassifierModel, MultiLabelClassifierModel, ResidueClassifierModel
from atomica.data.dataset import MultiClassLabelledPDBDataset, LabelledPDBDataset
from atomica.trainers import Trainer

from multiclass_metrics import compute_multiclass_metrics
from multilabel_metrics import compute_multilabel_metrics

from torch.utils.data import DataLoader
import torch
from tqdm import tqdm
import numpy as np
import pandas as pd
import os
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, f1_score

DATA_DIR="/n/holylfs06/LABS/mzitnik_lab/Lab/afang/ATOMICA/baselines/rnaglib_tasks"
MODEL_DIR="/n/netscratch/mzitnik_lab/Lab/afang/ATOMICA/baselines/rnaglib_benchmark"

# RNAGo

Compared to results in RNAGlib, this is SOTA! Yay!

In [ ]:
model_checkpoint = f"{MODEL_DIR}/RNAGo/models/version_17/checkpoint/epoch294_step45135.pt"

def get_model(model_checkpoint: str) -> str:
    model_config = os.path.join(os.path.dirname(model_checkpoint), "config.json")
    model = MultiLabelClassifierModel.load_from_config_and_weights(model_config, model_checkpoint)
    return model

dataset = MultiClassLabelledPDBDataset(f"{DATA_DIR}/RNAGo/RNAGo_test_processed.parquet")
model = get_model(model_checkpoint)

atomica_preds = []
model.eval()
model.to("cuda")
batch_size = 1
for i in tqdm(range(0, len(dataset), batch_size), total=len(dataset) // batch_size):
    with torch.no_grad():
        batch = [dataset[j] for j in range(i, min(i+batch_size, len(dataset)))]
        batch = MultiClassLabelledPDBDataset.collate_fn(batch)
        batch = Trainer.to_device(batch, "cuda")
        atomica_preds.append(model.infer(batch).cpu().numpy())
atomica_preds = np.concatenate(atomica_preds)
atomica_labels = np.array([x['label'] for x in dataset.data])
atomica_results = pd.DataFrame({
    'id': [x['id'] for x in dataset.data],
    'label': [atomica_labels[i] for i in range(len(atomica_labels))],
    'pred_probability': [atomica_preds[i] for i in range(len(atomica_preds))],
})

In [5]:
atomica_results

,id,label,pred_probability
0,6y50_2_1_188,"[0.0, 0.0, 1.0, 0.0, 0.0]","[2.2812487e-10, 6.480229e-14, 0.8928618, 1.295..."
1,6ff4_2_1_188,"[0.0, 0.0, 1.0, 0.0, 0.0]","[6.1552465e-11, 2.6185588e-14, 0.98908573, 1.8..."
2,7abh_2_1_188,"[0.0, 0.0, 1.0, 0.0, 0.0]","[1.9512479e-10, 6.045678e-14, 0.9175288, 1.369..."
3,5mps_5_55_178,"[1.0, 1.0, 0.0, 0.0, 0.0]","[0.99999726, 0.9956648, 1.4809575e-11, 0.00324..."
4,7q4o_2_1_188,"[0.0, 0.0, 1.0, 0.0, 0.0]","[6.1533983e-06, 2.7511915e-10, 0.0005391409, 1..."
...,...,...,...
70,4e8n_A_87_259,"[0.0, 0.0, 0.0, 0.0, 0.0]","[0.053795397, 2.5034537e-05, 1.4992314e-05, 0...."
71,4e8m_A_87_259,"[0.0, 0.0, 0.0, 0.0, 0.0]","[0.009880038, 6.2784035e-05, 0.0005812664, 0.0..."
72,6jq5_B_1_81,"[0.0, 0.0, 0.0, 0.0, 0.0]","[0.0014083969, 2.5739064e-08, 1.3523576e-05, 2..."
73,6jq5_A_1_81,"[0.0, 0.0, 0.0, 0.0, 0.0]","[0.13598688, 2.8769289e-06, 2.604722e-06, 0.00..."


In [3]:
atomica_preds.shape

(75, 5)

In [20]:
len(atomica_results)

75

In [10]:
np.mean(atomica_results['label'] == atomica_results['pred'])

0.7333333333333333

In [7]:
metrics = compute_multilabel_metrics(
    y_true=np.stack(atomica_results['label']),
    y_proba=np.stack(atomica_results['pred_probability']),
)

In [8]:
metrics

MultilabelMetricsResult(subset_accuracy=0.7333333333333333, f1_macro=0.828888888888889, f1_micro=0.7435897435897436, f1_weighted=0.7245862884160756, f1_samples=0.29777777777777775, jaccard_macro=0.7428571428571429, jaccard_micro=0.5918367346938775, jaccard_weighted=0.6015473887814313, jaccard_samples=0.29333333333333333, roc_auc_ovr_macro=0.9442170550038197, roc_auc_ovr_weighted=0.9013173427011036, roc_auc_ovr_micro=0.9335106382978723, per_label={0: {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'support': 8.0, 'jaccard': 1.0}, 1: {'precision': 1.0, 'recall': 0.7142857142857143, 'f1': 0.8333333333333334, 'support': 7.0, 'jaccard': 0.7142857142857143}, 2: {'precision': 1.0, 'recall': 0.6363636363636364, 'f1': 0.7777777777777778, 'support': 11.0, 'jaccard': 0.6363636363636364}, 3: {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'support': 1.0, 'jaccard': 1.0}, 4: {'precision': 0.8, 'recall': 0.4, 'f1': 0.5333333333333333, 'support': 20.0, 'jaccard': 0.36363636363636365}}, per_label_ovr_auc={0

# RNA Ligand

This is also SOTA! Yay!
The balanced sampling helps. Also need to ignore checkpoints in the first 10 epochs due to instability.

In [15]:
model_checkpoint = f"{MODEL_DIR}/RNA_Ligand/models/version_5/checkpoint/epoch227_step10032.pt"

def get_model(model_checkpoint: str) -> str:
    model_config = os.path.join(os.path.dirname(model_checkpoint), "config.json")
    model = MultiClassClassifierModel.load_from_config_and_weights(model_config, model_checkpoint)
    return model

dataset = MultiClassLabelledPDBDataset(f"{DATA_DIR}/RNA_Ligand/RNA_Ligand_test_processed.parquet")
model = get_model(model_checkpoint)

atomica_preds = []
model.eval()
model.to("cuda")
batch_size = 1
for i in tqdm(range(0, len(dataset), batch_size), total=len(dataset) // batch_size):
    with torch.no_grad():
        batch = [dataset[j] for j in range(i, min(i+batch_size, len(dataset)))]
        batch = MultiClassLabelledPDBDataset.collate_fn(batch)
        batch = Trainer.to_device(batch, "cuda")
        atomica_preds.append(model.infer(batch).cpu().numpy())
atomica_preds = np.concatenate(atomica_preds)
atomica_labels = np.array([x['label'] for x in dataset.data])
pred_indxes = np.argmax(atomica_preds, axis=1)
atomica_results = pd.DataFrame({
    'id': [x['id'] for x in dataset.data],
    'label': atomica_labels,
    'pred': pred_indxes,
    'pred_probability': [atomica_preds[i] for i in range(len(atomica_preds))],
    'seed': i,
})

/n/home13/afang/.conda/envs/interactenv/lib/python3.9/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
100%|██████████| 44/44 [00:10<00:00,  4.28it/s]


In [16]:
atomica_results['label'].value_counts()

label
1    29
0    10
2     5
Name: count, dtype: int64

In [17]:
np.mean(atomica_results['label'] == atomica_results['pred'])

0.7272727272727273

In [18]:
metrics = compute_multiclass_metrics(
    np.array(atomica_results['label']),
    np.array(atomica_results['pred']),
    np.stack(atomica_results['pred_probability']),
    [0,1,2],
)

In [19]:
metrics

MetricsResult(accuracy=0.7272727272727273, balanced_accuracy=0.5770114942528736, f1_macro=0.5656322843822844, f1_micro=0.7272727272727273, f1_weighted=0.6880214955499047, jaccard_macro=0.4288493038493038, jaccard_micro=0.5714285714285714, jaccard_weighted=0.5648941813714541, roc_auc_ovr_macro=0.7864140704911495, roc_auc_ovr_weighted=0.8365933086521321, roc_auc_ovo_macro=0.7637931034482759, roc_auc_ovo_weighted=0.7973942006269593, per_class={0: {'precision': 0.6666666666666666, 'recall': 0.2, 'f1': 0.3076923076923077, 'support': 10.0, 'jaccard': 0.18181818181818182}, 1: {'precision': 0.7714285714285715, 'recall': 0.9310344827586207, 'f1': 0.84375, 'support': 29.0, 'jaccard': 0.7297297297297297}, 2: {'precision': 0.5, 'recall': 0.6, 'f1': 0.5454545454545454, 'support': 5.0, 'jaccard': 0.375}}, per_class_ovr_auc={0: 0.5558823529411765, 1: 0.9264367816091953, 2: 0.8769230769230769})

# RNA-Protein
This also looks good

In [69]:
model_checkpoint = f"{MODEL_DIR}/RNA_Protein/models/version_1/checkpoint/epoch14_step3210.pt"

def get_model(model_checkpoint: str) -> str:
    model_config = os.path.join(os.path.dirname(model_checkpoint), "config.json")
    model = ResidueClassifierModel.load_from_config_and_weights(model_config, model_checkpoint)
    return model

dataset = LabelledPDBDataset(f"{DATA_DIR}/RNA_Protein/RNA_Protein_test_processed.parquet")
model = get_model(model_checkpoint)

atomica_preds = []
model.eval()
model.to("cuda")
batch_size = 1
for i in tqdm(range(0, len(dataset), batch_size), total=len(dataset) // batch_size):
    with torch.no_grad():
        batch = [dataset[j] for j in range(i, min(i+batch_size, len(dataset)))]
        batch = LabelledPDBDataset.collate_fn(batch)
        batch = Trainer.to_device(batch, "cuda")
        atomica_preds.append(model.infer(batch).cpu().numpy())
atomica_preds = np.concatenate(atomica_preds).flatten()
atomica_labels = np.concatenate([x['label'] for x in dataset.data])
atomica_ids = sum([[x['id']] * len(x['label']) for x in dataset.data], [])

atomica_results = pd.DataFrame({
    'id': atomica_ids,
    'label': atomica_labels,
    'pred': atomica_preds,
})
atomica_results

/n/home13/afang/.conda/envs/interactenv/lib/python3.9/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
100%|██████████| 190/190 [00:26<00:00,  7.06it/s]


,id,label,pred
0,1et4_0,0.0,0.547619
1,1et4_0,0.0,0.543388
2,1et4_0,0.0,0.544882
3,1et4_0,0.0,0.544413
4,1et4_0,0.0,0.526700
...,...,...,...
10846,9ij0_0,1.0,0.547497
10847,9ij0_0,0.0,0.543064
10848,9ij0_0,0.0,0.546277
10849,9ij0_0,0.0,0.547949


In [34]:
auroc = roc_auc_score(atomica_labels, atomica_preds)

precision, recall, thresholds = precision_recall_curve(atomica_labels, atomica_preds)
auprc = auc(recall, precision)

print("AUROC: ", auroc)
print("AUPRC: ", auprc)
print("Mean label: ", np.mean(atomica_labels))

AUROC:  0.747219629920314
AUPRC:  0.5440883585916663
Mean label:  0.274076122016404


In [70]:
thresholds = np.linspace(0.0, 1.0, 101)  # e.g. test thresholds from 0.00 to 1.00
f1s = [f1_score(atomica_labels, (atomica_preds >= t).astype(int)) for t in thresholds]

best_t = thresholds[np.argmax(f1s)]
best_f1 = max(f1s)

print(f"Best threshold = {best_t:.3f}, Best F1 = {best_f1:.4f}")

Best threshold = 0.220, Best F1 = 0.5430


### Grouped scores by id

In [71]:
atomica_results['pred_bin'] = atomica_preds >= best_t

atomica_results['correct'] = atomica_results['label'] == atomica_results['pred_bin']
balanced_acc = atomica_results.groupby('id')['correct'].mean().mean()

atomica_results['baseline_correct'] = atomica_results['label'] == 0
baseline_acc = atomica_results.groupby('id')['baseline_correct'].mean().mean()

print(balanced_acc, baseline_acc)

0.6928521605484866 0.6970169462442661


In [72]:
grouped = atomica_results.groupby('id')

auprcs, aurocs = [], []
baseline = []

for _, g in grouped:
    y_true = g['label']
    y_pred = g['pred']
    # skip if only one class is present (AUROC undefined)
    if len(set(y_true)) > 1:
        precision, recall, thresholds = precision_recall_curve(y_true, y_pred)
        auprc = auc(recall, precision)
        auroc = roc_auc_score(y_true, y_pred)
        auprcs.append(auprc)
        aurocs.append(auroc)
        baseline.append(np.mean(y_true))


mean_auprc = sum(auprcs) / len(auprcs)
mean_auroc = sum(aurocs) / len(aurocs)
mean_baseline = sum(baseline) / len(baseline)

print(f"Mean AUPRC: {mean_auprc:.4f}")
print(f"Mean AUROC: {mean_auroc:.4f}")
print(f"Baseline: {mean_baseline:.4f}")

Mean AUPRC: 0.5396
Mean AUROC: 0.5999
Baseline: 0.4230


# RNA-Site
This also looks good

In [ ]:
model_checkpoint = f"{MODEL_DIR}/RNA_Site/models/version_4/checkpoint/epoch82_step3901.pt"

def get_model(model_checkpoint: str) -> str:
    model_config = os.path.join(os.path.dirname(model_checkpoint), "config.json")
    model = ResidueClassifierModel.load_from_config_and_weights(model_config, model_checkpoint)
    return model

dataset = LabelledPDBDataset(f"{DATA_DIR}/RNA_Site/RNA_Site_test_processed.parquet")
model = get_model(model_checkpoint)

atomica_preds = []
model.eval()
model.to("cuda")
batch_size = 1
for i in tqdm(range(0, len(dataset), batch_size), total=len(dataset) // batch_size):
    with torch.no_grad():
        batch = [dataset[j] for j in range(i, min(i+batch_size, len(dataset)))]
        batch = LabelledPDBDataset.collate_fn(batch)
        batch = Trainer.to_device(batch, "cuda")
        atomica_preds.append(model.infer(batch).cpu().numpy())
atomica_preds = np.concatenate(atomica_preds).flatten()
atomica_labels = np.concatenate([x['label'] for x in dataset.data])
atomica_ids = sum([[x['id']] * len(x['label']) for x in dataset.data], [])

atomica_results = pd.DataFrame({
    'id': atomica_ids,
    'label': atomica_labels,
    'pred': atomica_preds,
})
atomica_results

In [41]:
auroc = roc_auc_score(atomica_labels, atomica_preds)

precision, recall, thresholds = precision_recall_curve(atomica_labels, atomica_preds)
auprc = auc(recall, precision)

f1 = f1_score(atomica_labels, atomica_preds >= 0.3)

print("AUROC: ", auroc)
print("AUPRC: ", auprc)
print("F1: ", f1)
print("Mean label: ", np.mean(atomica_labels))

AUROC:  0.6306351956585701
AUPRC:  0.17276746516192792
F1:  0.22456140350877193
Mean label:  0.07824074074074074


In [43]:
thresholds = np.linspace(0.0, 1.0, 101)  # e.g. test thresholds from 0.00 to 1.00
f1s = [f1_score(atomica_labels, (atomica_preds >= t).astype(int)) for t in thresholds]

best_t = thresholds[np.argmax(f1s)]
best_f1 = max(f1s)

print(f"Best threshold = {best_t:.3f}, Best F1 = {best_f1:.4f}")

Best threshold = 0.480, Best F1 = 0.2367


### Grouped scores by id

In [62]:
atomica_results['pred_bin'] = atomica_preds >= best_t

atomica_results['correct'] = atomica_results['label'] == atomica_results['pred_bin']
balanced_acc = atomica_results.groupby('id')['correct'].mean().mean()

atomica_results['baseline_correct'] = atomica_results['label'] == 0
baseline_acc = atomica_results.groupby('id')['baseline_correct'].mean().mean()

print(balanced_acc, baseline_acc)

0.8900087226463383 0.8965912059076241


In [67]:
grouped = atomica_results.groupby('id')

auprcs, aurocs = [], []
baseline = []

for _, g in grouped:
    y_true = g['label']
    y_pred = g['pred']
    # skip if only one class is present (AUROC undefined)
    if len(set(y_true)) > 1:
        precision, recall, thresholds = precision_recall_curve(y_true, y_pred)
        auprc = auc(recall, precision)
        auroc = roc_auc_score(y_true, y_pred)
        auprcs.append(auprc)
        aurocs.append(auroc)
        baseline.append(np.mean(y_true))


mean_auprc = sum(auprcs) / len(auprcs)
mean_auroc = sum(aurocs) / len(aurocs)
mean_baseline = sum(baseline) / len(baseline)

print(f"Mean AUPRC: {mean_auprc:.4f}")
print(f"Mean AUROC: {mean_auroc:.4f}")
print(f"Baseline: {mean_baseline:.4f}")

Mean AUPRC: 0.2359
Mean AUROC: 0.6016
Baseline: 0.1034
